In [1]:
from typing import TypedDict,Annotated
from langgraph.types import Send
import operator
from langgraph.graph import START,END,StateGraph,MessagesState
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

True

In [5]:
class GlobalState(TypedDict):
    items: list
    results: Annotated[list,operator.add]

class ItemState(TypedDict):
    item: str

def process_state(state:ItemState)->dict:
    item=state['item']
    print(f"Processing Data : {state['item']}")
    result=f"Processed:{state['item']}"
    return {"results": [result]}

def distribute_load(state:GlobalState)->dict:
    print(f"Distributing {state['items']} loads for processing")
    return [Send("process",{"item":item}) for item in state['items']]

graph=StateGraph(GlobalState)
graph.add_node("process",process_state)
graph.add_conditional_edges(START,distribute_load,["process"])
graph.add_edge("process",END)

app=graph.compile()
result=app.invoke({
    "items":["apple", "banana", "cherry", "date"],
    "results":[]
})

print(result)


Distributing ['apple', 'banana', 'cherry', 'date'] loads for processing
Processing Data : apple
Processing Data : banana
Processing Data : cherry
Processing Data : date
{'items': ['apple', 'banana', 'cherry', 'date'], 'results': ['Processed:apple', 'Processed:banana', 'Processed:cherry', 'Processed:date']}


In [18]:
class GlobalState(TypedDict):
    documents: list[str]
    reviews: Annotated[list[str],operator.add]
    summary: str

class NodeState(TypedDict):
    documents: str
    doc_index: int

def load_documents(state:GlobalState)->dict:
    """Loads 4 documents into Global State:"""
    docs=[
         "Q4 Revenue Report: revenue up 18% YoY",
        "Risk Assessment: supply chain risks identified",
        "Product Roadmap: 3 major features planned for H1",
        "HR Update: headcount grew by 15 in Q4",
    ]
    print(f"Documents of {len(docs)} are loaded")
    return {"documents":docs}

def review_documents(state:NodeState)->dict:
    """
    Worker node — receives ONE document via Send().
    Note: 'state' here is WorkerState, NOT OverallState.
    Each parallel invocation of this node has a different 'document'.
    """

    doc=state['documents']
    index=state['doc_index']
    review=f"Reviewing DOC {index+1}: {doc[:40]}"
    print(f"DOCS {index+1} Reviewed")
    return {"reviews":[review]}

def summary_creation(state:GlobalState)->dict:
    """Fan-in node: receives all reviews merged by the reducer."""
    print(f"Documents collected : {len(state['reviews'])} , Generating summary ...")
    summary="Summary :".join(state['reviews'])

    return {"summary":summary}

def dispatch_reviewers(state:GlobalState)->dict:
    """Called as a conditional edge function.
    Returns N Send objects → spawns N parallel review_document workers,
    each with its OWN custom state dict (different document + index).
    """
    return [Send("review",{"documents":doc,"doc_index":i}) for i,doc in enumerate(state['documents'])]

graph=StateGraph(GlobalState)
graph.add_node("load",load_documents)
graph.add_node("review",review_documents)
graph.add_node("summary",summary_creation)
graph.add_node("dispatch",dispatch_reviewers)

graph.add_edge(START,"load")
graph.add_conditional_edges("load",dispatch_reviewers,["review"])
graph.add_edge("review","summary")
graph.add_edge("summary",END)

app=graph.compile()

result=app.invoke({"documents":[],"reviews":[],"summary":""})
print(result)

Documents of 4 are loaded
DOCS 1 Reviewed
DOCS 2 Reviewed
DOCS 3 Reviewed
DOCS 4 Reviewed
Documents collected : 4 , Generating summary ...
{'documents': ['Q4 Revenue Report: revenue up 18% YoY', 'Risk Assessment: supply chain risks identified', 'Product Roadmap: 3 major features planned for H1', 'HR Update: headcount grew by 15 in Q4'], 'reviews': ['Reviewing DOC 1: Q4 Revenue Report: revenue up 18% YoY', 'Reviewing DOC 2: Risk Assessment: supply chain risks iden', 'Reviewing DOC 3: Product Roadmap: 3 major features planne', 'Reviewing DOC 4: HR Update: headcount grew by 15 in Q4'], 'summary': 'Reviewing DOC 1: Q4 Revenue Report: revenue up 18% YoYSummary :Reviewing DOC 2: Risk Assessment: supply chain risks idenSummary :Reviewing DOC 3: Product Roadmap: 3 major features planneSummary :Reviewing DOC 4: HR Update: headcount grew by 15 in Q4'}


['Q4 Revenue Report: revenue up 18% YoY', 'Risk Assessment: supply chain risks identified', 'Product Roadmap: 3 major features planned for H1', 'HR Update: headcount grew by 15 in Q4']
